# Experiment: Resume IR PDF Comparison

Objective:
- Load the persisted IR JSON from the first notebook.
- Reconstruct LaTeX from the saved IR.
- Compile the original and reconstructed LaTeX sources into PDFs.
- Compare the PDFs visually page by page.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
import json
import pprint
import shutil
import subprocess
from typing import Any, Optional

from PIL import Image, ImageChops
from pypdf import PdfReader


def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "tmp" / "resume_ir_roundtrip" / "resume_ir.json").exists():
            return candidate
        if (candidate / "templates" / "resume_templates" / "data_science_resume.tex").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
ARTIFACT_DIR = REPO_ROOT / "tmp" / "resume_ir_roundtrip"
IR_PATH = ARTIFACT_DIR / "resume_ir.json"
assert IR_PATH.exists(), IR_PATH
ARTIFACT_DIR


In [ ]:
class EscapingMode(str, Enum):
    TRUSTED_LATEX = "trusted_latex"
    PLAIN_TEXT = "plain_text"


class Provenance(str, Enum):
    ACTIVE = "active"
    RAW_FALLBACK = "raw_fallback"
    GENERATED = "generated"


class RenderMode(str, Enum):
    SOURCE_PRESERVED = "source_preserved"
    REGENERATED = "regenerated"


class SectionKind(str, Enum):
    HIGHLIGHTS = "highlights"
    SKILLS = "skills"
    EXPERIENCE = "experience"
    PROJECTS = "projects"
    EDUCATION = "education"
    PUBLICATIONS = "publications"
    CUSTOM = "custom"


@dataclass(slots=True)
class TextField:
    value: str
    mode: EscapingMode = EscapingMode.TRUSTED_LATEX


@dataclass(slots=True)
class BulletIR:
    id: str
    text: TextField
    raw_text: str


@dataclass(slots=True)
class EntryIR:
    id: str
    title: Optional[TextField]
    organization: Optional[TextField]
    location: Optional[TextField]
    date_range: Optional[TextField]
    bullets: list[BulletIR] = field(default_factory=list)
    description: Optional[TextField] = None
    hyperlink: Optional[str] = None
    raw_source: str = ""
    render_mode: RenderMode = RenderMode.SOURCE_PRESERVED
    warnings: list[str] = field(default_factory=list)


@dataclass(slots=True)
class SkillsCategoryIR:
    id: str
    name: TextField
    items: list[TextField] = field(default_factory=list)
    raw_source: str = ""


@dataclass(slots=True)
class SectionIR:
    id: str
    kind: SectionKind
    title: str
    order: int
    provenance: Provenance
    raw_source: str
    description: Optional[TextField] = None
    bullets: list[BulletIR] = field(default_factory=list)
    entries: list[EntryIR] = field(default_factory=list)
    skills_categories: list[SkillsCategoryIR] = field(default_factory=list)
    render_mode: RenderMode = RenderMode.SOURCE_PRESERVED
    warnings: list[str] = field(default_factory=list)


@dataclass(slots=True)
class HeaderIR:
    name: Optional[str]
    emails: list[str] = field(default_factory=list)
    phone_numbers: list[str] = field(default_factory=list)
    location: Optional[str] = None
    links: list[dict[str, str]] = field(default_factory=list)
    raw_header_block: str = ""


@dataclass(slots=True)
class DocumentMetaIR:
    source_path: str
    preamble: str
    postamble: str
    parser_warnings: list[str] = field(default_factory=list)
    original_text: str = ""
    original_section_ids: list[str] = field(default_factory=list)


@dataclass(slots=True)
class ResumeIR:
    document_meta: DocumentMetaIR
    header: HeaderIR
    sections: list[SectionIR]


def text_field_from_dict(data: Optional[dict[str, Any]]) -> Optional[TextField]:
    if data is None:
        return None
    return TextField(value=data["value"], mode=EscapingMode(data["mode"]))


def bullet_from_dict(data: dict[str, Any]) -> BulletIR:
    return BulletIR(id=data["id"], text=text_field_from_dict(data["text"]), raw_text=data["raw_text"])


def entry_from_dict(data: dict[str, Any]) -> EntryIR:
    return EntryIR(
        id=data["id"],
        title=text_field_from_dict(data.get("title")),
        organization=text_field_from_dict(data.get("organization")),
        location=text_field_from_dict(data.get("location")),
        date_range=text_field_from_dict(data.get("date_range")),
        bullets=[bullet_from_dict(item) for item in data.get("bullets", [])],
        description=text_field_from_dict(data.get("description")),
        hyperlink=data.get("hyperlink"),
        raw_source=data.get("raw_source", ""),
        render_mode=RenderMode(data.get("render_mode", RenderMode.SOURCE_PRESERVED.value)),
        warnings=list(data.get("warnings", [])),
    )


def skills_category_from_dict(data: dict[str, Any]) -> SkillsCategoryIR:
    return SkillsCategoryIR(
        id=data["id"],
        name=text_field_from_dict(data["name"]),
        items=[text_field_from_dict(item) for item in data.get("items", [])],
        raw_source=data.get("raw_source", ""),
    )


def section_from_dict(data: dict[str, Any]) -> SectionIR:
    return SectionIR(
        id=data["id"],
        kind=SectionKind(data["kind"]),
        title=data["title"],
        order=data["order"],
        provenance=Provenance(data["provenance"]),
        raw_source=data["raw_source"],
        description=text_field_from_dict(data.get("description")),
        bullets=[bullet_from_dict(item) for item in data.get("bullets", [])],
        entries=[entry_from_dict(item) for item in data.get("entries", [])],
        skills_categories=[skills_category_from_dict(item) for item in data.get("skills_categories", [])],
        render_mode=RenderMode(data.get("render_mode", RenderMode.SOURCE_PRESERVED.value)),
        warnings=list(data.get("warnings", [])),
    )


def resume_from_dict(data: dict[str, Any]) -> ResumeIR:
    document_meta = data["document_meta"]
    header = data["header"]
    return ResumeIR(
        document_meta=DocumentMetaIR(
            source_path=document_meta["source_path"],
            preamble=document_meta["preamble"],
            postamble=document_meta["postamble"],
            parser_warnings=list(document_meta.get("parser_warnings", [])),
            original_text=document_meta.get("original_text", ""),
            original_section_ids=list(document_meta.get("original_section_ids", [])),
        ),
        header=HeaderIR(
            name=header.get("name"),
            emails=list(header.get("emails", [])),
            phone_numbers=list(header.get("phone_numbers", [])),
            location=header.get("location"),
            links=list(header.get("links", [])),
            raw_header_block=header.get("raw_header_block", ""),
        ),
        sections=[section_from_dict(item) for item in data.get("sections", [])],
    )


def escape_latex(text: str) -> str:
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in text)


def render_text_field(field_value: Optional[TextField]) -> str:
    if field_value is None:
        return ""
    if field_value.mode == EscapingMode.PLAIN_TEXT:
        return escape_latex(field_value.value)
    return field_value.value


def render_bullet_list(bullets: list[BulletIR], spacing: str = "-4pt") -> str:
    lines = [r"\begin{itemize}[leftmargin=0.6cm, label={\textbullet}]"]
    for bullet in bullets:
        lines.append(f"    \\item {render_text_field(bullet.text)}")
        lines.append(f"    \\vspace{{{spacing}}}")
    lines.append(r"\end{itemize}")
    return "\n".join(lines)


def render_entry_title(entry: EntryIR) -> str:
    title = render_text_field(entry.title)
    if entry.hyperlink:
        return f"\\href{{{entry.hyperlink}}}{{{title}}}"
    return title


def render_regenerated_entry(entry: EntryIR, section_kind: SectionKind) -> str:
    date_tex = render_text_field(entry.date_range)
    title_tex = render_entry_title(entry)
    organization_tex = render_text_field(entry.organization)
    location_tex = render_text_field(entry.location)

    if section_kind == SectionKind.EXPERIENCE:
        body_lines: list[str] = [r"\vspace{4pt}"]
        if organization_tex and location_tex:
            body_lines.append(f"{organization_tex} \\hfill \\textit{{{location_tex}}}")
        elif organization_tex:
            body_lines.append(organization_tex)
        elif location_tex:
            body_lines.append(f"\\textit{{{location_tex}}}")
        if entry.description or entry.bullets:
            body_lines.append(r"\vspace{2pt}")
        if entry.description:
            body_lines.append(render_text_field(entry.description))
        if entry.bullets:
            body_lines.append(render_bullet_list(entry.bullets, spacing="-4pt"))
        body = "\n".join(line for line in body_lines if line).strip()
        return (
            "\\customcventry"
            f"{{{date_tex}}}{{}}{{{title_tex}}}{{}}{{}}"
            + "{\n" + body + "\n}"
        )

    if section_kind == SectionKind.EDUCATION:
        body_lines: list[str] = []
        if entry.description or entry.bullets:
            body_lines.append(r"\vspace{2pt}")
        if entry.description:
            body_lines.append(render_text_field(entry.description))
        if entry.bullets:
            body_lines.append(render_bullet_list(entry.bullets, spacing="-4pt"))
        body = "\n".join(line for line in body_lines if line).strip()
        return (
            "\\customcventry"
            f"{{{date_tex}}}{{{{{organization_tex}}}}}{{{title_tex}}}{{{location_tex}}}{{}}{{}}"
            + "{\n" + body + "\n}"
        )

    if section_kind == SectionKind.PROJECTS:
        body_lines: list[str] = []
        if entry.description or entry.bullets:
            body_lines.append(r"\vspace{2pt}")
        if entry.description:
            body_lines.append(render_text_field(entry.description))
        if entry.bullets:
            body_lines.append(render_bullet_list(entry.bullets, spacing="-2pt"))
        body = "\n".join(line for line in body_lines if line).strip()
        return (
            "\\customcventry"
            f"{{{date_tex}}}{{}}{{{title_tex}}}{{}}{{}}"
            + "{\n" + body + "\n}"
        )

    body_parts: list[str] = []
    if entry.description:
        body_parts.append(r"\vspace{2pt}")
        body_parts.append(render_text_field(entry.description))
    if entry.bullets:
        if not body_parts:
            body_parts.append(r"\vspace{2pt}")
        body_parts.append(render_bullet_list(entry.bullets, spacing="-4pt"))
    body = "\n".join(part for part in body_parts if part).strip()
    return (
        "\\customcventry"
        f"{{{date_tex}}}"
        f"{{{organization_tex}}}"
        f"{{{title_tex}}}"
        f"{{{location_tex}}}"
        "{}"
        f"{{{body}}}"
    )


def render_section(section: SectionIR) -> str:
    if section.render_mode == RenderMode.SOURCE_PRESERVED and section.raw_source.strip():
        return section.raw_source.strip("\n")
    if section.kind == SectionKind.CUSTOM:
        return section.raw_source.strip()
    lines = [f"\\section{{{section.title}}}"]
    if section.kind == SectionKind.HIGHLIGHTS:
        lines.extend([r"\setlength{\itemsep}{0pt}", r"\setlength{\parskip}{0pt}", render_bullet_list(section.bullets, spacing="-4pt")])
    elif section.kind == SectionKind.SKILLS:
        lines.extend([r"\vspace{-4pt}", r"\begin{tabularx}{\textwidth}{>{\raggedright\arraybackslash\bfseries}p{5.5cm}@{\hspace{0cm}}X}"])
        for category in section.skills_categories:
            rendered_items = ", ".join(render_text_field(item) for item in category.items)
            lines.append(f"{render_text_field(category.name)} & {rendered_items} \\\\[4pt]")
        lines.append(r"\end{tabularx}")
        lines.append(r"\vspace{-16pt}")
    elif section.kind in {SectionKind.EXPERIENCE, SectionKind.EDUCATION, SectionKind.PROJECTS}:
        spacer = r"\vspace{6pt}" if section.kind != SectionKind.EDUCATION else r"\vspace{10pt}"
        rendered_entries = []
        for entry in section.entries:
            if entry.render_mode == RenderMode.SOURCE_PRESERVED and entry.raw_source.strip():
                rendered_entries.append(entry.raw_source.strip("\n"))
            else:
                rendered_entries.append(render_regenerated_entry(entry, section.kind))
        lines.append("\n\n".join(rendered_entries))
        lines.append(spacer)
    elif section.kind == SectionKind.PUBLICATIONS:
        lines.extend([r"\setlength{\itemsep}{0pt}", r"\setlength{\parskip}{0pt}"])
        if section.description:
            lines.append(render_text_field(section.description))
        if section.bullets:
            lines.append(render_bullet_list(section.bullets, spacing="-4pt"))
    return "\n".join(line for line in lines if line).strip()


def can_render_from_original_source(resume: ResumeIR) -> bool:
    if not resume.document_meta.original_text:
        return False
    if any(section.render_mode != RenderMode.SOURCE_PRESERVED for section in resume.sections):
        return False
    current_section_ids = [section.id for section in resume.sections]
    return current_section_ids == resume.document_meta.original_section_ids


def render_resume(resume: ResumeIR) -> str:
    if can_render_from_original_source(resume):
        return resume.document_meta.original_text

    parts = [resume.document_meta.preamble.rstrip(), r"\begin{document}"]
    if resume.header.raw_header_block:
        parts.append(resume.header.raw_header_block.strip())
    for section in resume.sections:
        parts.append(render_section(section))
    parts.append(r"\end{document}")
    if resume.document_meta.postamble.strip():
        parts.append(resume.document_meta.postamble.strip())
    return "\n\n".join(part for part in parts if part).strip() + "\n"


def pretty_json(payload: Any) -> str:
    return json.dumps(payload, indent=2, ensure_ascii=False)


In [ ]:
resume_ir_payload = json.loads(IR_PATH.read_text(encoding="utf-8"))
resume_ir = resume_from_dict(resume_ir_payload)
source_resume_path = Path(resume_ir.document_meta.source_path)
original_source_text = resume_ir.document_meta.original_text or source_resume_path.read_text(encoding="utf-8")
reconstructed_resume_text = render_resume(resume_ir)

original_tex_path = ARTIFACT_DIR / "original_source_copy.tex"
reconstructed_tex_path = ARTIFACT_DIR / "reconstructed_resume.tex"
original_tex_path.write_text(original_source_text, encoding="utf-8")
reconstructed_tex_path.write_text(reconstructed_resume_text, encoding="utf-8")

pprint.pp({
    "artifact_dir": str(ARTIFACT_DIR),
    "original_tex": str(original_tex_path),
    "reconstructed_tex": str(reconstructed_tex_path),
    "exact_tex_match": original_source_text == reconstructed_resume_text,
})


In [ ]:
def compile_latex(tex_path: Path, target_pdf_path: Path) -> dict[str, Any]:
    command = [
        "latexmk",
        "-quiet",
        "-pdf",
        "-interaction=nonstopmode",
        "-halt-on-error",
        tex_path.name,
    ]
    result = subprocess.run(
        command,
        cwd=tex_path.parent,
        text=True,
        capture_output=True,
    )
    generated_pdf = tex_path.with_suffix(".pdf")
    log_path = tex_path.with_suffix(".log")
    success = result.returncode == 0 and generated_pdf.exists()
    if success:
        shutil.copy2(generated_pdf, target_pdf_path)
    return {
        "tex_path": str(tex_path),
        "target_pdf_path": str(target_pdf_path),
        "generated_pdf": str(generated_pdf),
        "log_path": str(log_path),
        "success": success,
        "returncode": result.returncode,
        "stdout_tail": result.stdout[-2000:],
        "stderr_tail": result.stderr[-2000:],
    }


original_compile = compile_latex(original_tex_path, ARTIFACT_DIR / "original.pdf")
reconstructed_compile = compile_latex(reconstructed_tex_path, ARTIFACT_DIR / "reconstructed.pdf")

compile_summary = {
    "original_compile": original_compile,
    "reconstructed_compile": reconstructed_compile,
}
pprint.pp({
    "original_compile_success": original_compile["success"],
    "reconstructed_compile_success": reconstructed_compile["success"],
    "original_log_path": original_compile["log_path"],
    "reconstructed_log_path": reconstructed_compile["log_path"],
})


In [ ]:
def rasterize_pdf(pdf_path: Path, stem: str, dpi: int = 200) -> list[Path]:
    raster_prefix = ARTIFACT_DIR / f"{stem}-raster"
    command = ["pdftoppm", "-png", "-r", str(dpi), str(pdf_path), str(raster_prefix)]
    subprocess.run(command, check=True, text=True, capture_output=True)
    raster_paths = sorted(ARTIFACT_DIR.glob(f"{stem}-raster-*.png"))
    normalized_paths: list[Path] = []
    for page_index, raster_path in enumerate(raster_paths, start=1):
        normalized_path = ARTIFACT_DIR / f"page-{page_index:03d}-{stem}.png"
        raster_path.replace(normalized_path)
        normalized_paths.append(normalized_path)
    return normalized_paths


comparison_summary: dict[str, Any] = {
    "compile": compile_summary,
    "pdfs_ready": original_compile["success"] and reconstructed_compile["success"],
    "page_count_match": False,
    "page_size_match": False,
    "pages": [],
}

if comparison_summary["pdfs_ready"]:
    original_pdf_path = ARTIFACT_DIR / "original.pdf"
    reconstructed_pdf_path = ARTIFACT_DIR / "reconstructed.pdf"
    original_reader = PdfReader(str(original_pdf_path))
    reconstructed_reader = PdfReader(str(reconstructed_pdf_path))

    original_page_count = len(original_reader.pages)
    reconstructed_page_count = len(reconstructed_reader.pages)
    comparison_summary["original_page_count"] = original_page_count
    comparison_summary["reconstructed_page_count"] = reconstructed_page_count
    comparison_summary["page_count_match"] = original_page_count == reconstructed_page_count

    page_sizes = []
    for page_index in range(min(original_page_count, reconstructed_page_count)):
        original_page = original_reader.pages[page_index]
        reconstructed_page = reconstructed_reader.pages[page_index]
        original_size = [float(original_page.mediabox.width), float(original_page.mediabox.height)]
        reconstructed_size = [float(reconstructed_page.mediabox.width), float(reconstructed_page.mediabox.height)]
        page_sizes.append({
            "page": page_index + 1,
            "original_size": original_size,
            "reconstructed_size": reconstructed_size,
            "size_match": original_size == reconstructed_size,
        })
    comparison_summary["page_sizes"] = page_sizes
    comparison_summary["page_size_match"] = all(item["size_match"] for item in page_sizes) and comparison_summary["page_count_match"]

    original_pngs = rasterize_pdf(original_pdf_path, "original")
    reconstructed_pngs = rasterize_pdf(reconstructed_pdf_path, "reconstructed")

    for page_index, (original_png, reconstructed_png) in enumerate(zip(original_pngs, reconstructed_pngs), start=1):
        with Image.open(original_png) as original_image, Image.open(reconstructed_png) as reconstructed_image:
            original_rgb = original_image.convert("RGB")
            reconstructed_rgb = reconstructed_image.convert("RGB")
            diff_image = ImageChops.difference(original_rgb, reconstructed_rgb)
            diff_bbox = diff_image.getbbox()
            diff_path = ARTIFACT_DIR / f"page-{page_index:03d}-diff.png"
            diff_image.save(diff_path)
            grayscale = diff_image.convert("L")
            histogram = grayscale.histogram()
            total_pixels = grayscale.width * grayscale.height
            identical_pixels = histogram[0]
            differing_pixels = total_pixels - identical_pixels
            comparison_summary["pages"].append({
                "page": page_index,
                "image_size": [grayscale.width, grayscale.height],
                "identical_pixels": int(identical_pixels),
                "differing_pixels": int(differing_pixels),
                "visually_identical": differing_pixels == 0,
                "diff_bbox": list(diff_bbox) if diff_bbox else None,
                "original_png": str(original_png),
                "reconstructed_png": str(reconstructed_png),
                "diff_png": str(diff_path),
            })
else:
    comparison_summary["error"] = "Skipping PDF comparison because at least one LaTeX compile failed."

(ARTIFACT_DIR / "comparison_summary.json").write_text(pretty_json(comparison_summary) + "\n", encoding="utf-8")

visual_summary = {
    "pdfs_ready": comparison_summary["pdfs_ready"],
    "page_count_match": comparison_summary.get("page_count_match"),
    "page_size_match": comparison_summary.get("page_size_match"),
    "visually_identical_pages": sum(1 for page in comparison_summary.get("pages", []) if page["visually_identical"]),
    "differing_pages": sum(1 for page in comparison_summary.get("pages", []) if not page["visually_identical"]),
    "comparison_summary_path": str(ARTIFACT_DIR / "comparison_summary.json"),
}
pprint.pp(visual_summary)
print(pretty_json(comparison_summary))


## Results

- `exact_tex_match` should now be `True` for unchanged input because the renderer preserves the original source when nothing in the IR has been modified.
- The PDF comparison remains the fidelity guardrail for catching any accidental drift after JSON save/load or future edit flows.
- If compilation fails, inspect the saved `.log` paths in the compile summary.
